# Day 4a: M3OT Dataset Inspection (diagnostic only)

**Purpose:** Download M3OT dataset (10.7 GB ZIP from Figshare), inspect structure, and print diagnostic info. Do NOT train yet.

**Why this step exists:** M3OT structure isn't documented beyond "MOT format + COCO JSON". I need to see the actual directory tree, annotation format, class list, and image dimensions before writing the processor for Day 4b.

**What to do:** Run all cells. Copy the full output of each cell and paste back to me. I'll finalize the real training notebook based on what we find.

**Expected runtime:** ~15-30 min (10 GB download dominates).

## CELL 1 -- Mount Drive (for caching the big download)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/SanjayMK2/datasets', exist_ok=True)
!df -h /content/drive/MyDrive 2>/dev/null | head -3 || df -h /content | head -3

## CELL 2 -- Download M3OT ZIP (10.7 GB, ~10-15 min)

Direct download from Figshare (verified via `curl https://api.figshare.com/v2/articles/28308887`).

Caches to Drive so re-running doesn't re-download.

In [ ]:
import os

CACHE = '/content/drive/MyDrive/SanjayMK2/datasets/M3OT.zip'
LOCAL = '/content/M3OT.zip'
URL = 'https://ndownloader.figshare.com/files/52023875'

if os.path.exists(CACHE):
    size_mb = os.path.getsize(CACHE) / 1e6
    print(f'Cached on Drive: {size_mb:.0f} MB. Copying to /content/...')
    !cp {CACHE} {LOCAL}
else:
    print('Downloading M3OT.zip (10.7 GB)...')
    !wget -O {LOCAL} -q --show-progress {URL}
    print('Caching to Drive (for future runs)...')
    !cp {LOCAL} {CACHE}

print(f'\nFile size: {os.path.getsize(LOCAL) / 1e9:.2f} GB')

## CELL 3 -- Inspect ZIP contents (no extraction)

Lists the first 50 entries in the ZIP, plus total file count by type.
This tells us the directory structure without extracting 10 GB.

In [ ]:
import zipfile
from collections import Counter

LOCAL = '/content/M3OT.zip'

with zipfile.ZipFile(LOCAL, 'r') as z:
    names = z.namelist()

print(f'Total entries in ZIP: {len(names)}')
print()
print('=== First 50 entries (directory tree preview) ===')
for n in names[:50]:
    print(f'  {n}')

print()
print('=== File type distribution ===')
exts = Counter()
for n in names:
    if n.endswith('/'):
        continue
    ext = n.split('.')[-1].lower() if '.' in n else 'noext'
    exts[ext] += 1
for ext, count in exts.most_common():
    print(f'  .{ext}: {count}')

print()
print('=== Top-level directory names ===')
top = set()
for n in names:
    top.add(n.split('/')[0])
for t in sorted(top)[:30]:
    print(f'  {t}/')

## CELL 4 -- Extract small files only (JSON annotations, README, etc.)

Skip the 10 GB of images. Just pull out text/JSON annotation files.

In [ ]:
import zipfile
import os

LOCAL = '/content/M3OT.zip'
EXTRACT_DIR = '/content/m3ot_inspect'
os.makedirs(EXTRACT_DIR, exist_ok=True)

# Only extract JSON, TXT, YAML, README files
text_exts = ('.json', '.txt', '.yaml', '.yml', '.md', '.csv')

with zipfile.ZipFile(LOCAL, 'r') as z:
    extracted = []
    for n in z.namelist():
        if n.endswith('/'):
            continue
        if any(n.lower().endswith(ext) for ext in text_exts):
            info = z.getinfo(n)
            # Skip huge files (some JSONs can be >100 MB, that's a sign they're the whole coco)
            # We want to see them but don't overload
            if info.file_size > 200_000_000:  # 200 MB
                print(f'  SKIP (too large, {info.file_size/1e6:.0f} MB): {n}')
                continue
            z.extract(n, EXTRACT_DIR)
            extracted.append((n, info.file_size))

print(f'Extracted {len(extracted)} text files')
for n, sz in extracted[:30]:
    print(f'  {sz/1024:>8.1f} KB  {n}')

## CELL 5 -- Dump annotation file structure

Look inside JSON/TXT files to find class names, annotation format.

In [ ]:
import json
from pathlib import Path

EXTRACT = Path('/content/m3ot_inspect')

# Look at YAML/TXT first (readme, config)
for ext in ['.yaml', '.yml', '.md', '.txt']:
    for f in EXTRACT.rglob(f'*{ext}'):
        if f.stat().st_size > 10_000:  # skip huge txts (could be MOT format)
            continue
        print(f'=== {f.relative_to(EXTRACT)} ({f.stat().st_size} bytes) ===')
        try:
            print(f.read_text()[:2000])
        except Exception as e:
            print(f'  read error: {e}')
        print()

# Look at small JSONs (likely categories, metadata)
print('\n=== JSON files ===')
for f in sorted(EXTRACT.rglob('*.json'), key=lambda x: x.stat().st_size):
    sz = f.stat().st_size
    print(f'\n--- {f.relative_to(EXTRACT)} ({sz/1024:.1f} KB) ---')
    if sz > 5_000_000:
        # For large JSONs, just show structure
        try:
            with open(f) as fp:
                data = json.load(fp)
            if isinstance(data, dict):
                print(f'  Top-level keys: {list(data.keys())}')
                for k, v in data.items():
                    if isinstance(v, list):
                        print(f'  {k}: list of {len(v)} items')
                        if v and isinstance(v[0], dict):
                            print(f'    Sample [0]: {v[0]}')
                    elif isinstance(v, dict):
                        print(f'  {k}: dict with keys {list(v.keys())[:10]}')
                    else:
                        print(f'  {k}: {type(v).__name__} = {str(v)[:100]}')
        except Exception as e:
            print(f'  parse error: {e}')
    else:
        try:
            content = f.read_text()
            print(content[:2000])
            if len(content) > 2000:
                print(f'  ... ({len(content) - 2000} more chars)')
        except Exception as e:
            print(f'  read error: {e}')

## CELL 6 -- Sample one image to check format

Extract a single image file from the ZIP to verify:
- Image format (JPG/PNG/TIFF)
- Dimensions
- Whether RGB and IR are separate files or interleaved

In [ ]:
import zipfile
import os
from PIL import Image

LOCAL = '/content/M3OT.zip'
SAMPLE_DIR = '/content/m3ot_sample'
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Extract the first image of each type
with zipfile.ZipFile(LOCAL, 'r') as z:
    names = z.namelist()
    # Find image candidates
    image_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')
    imgs = [n for n in names if n.lower().endswith(image_exts)]

print(f'Total image files in ZIP: {len(imgs)}')
print(f'First 10 image paths:')
for i in imgs[:10]:
    print(f'  {i}')

# Extract up to 3 sample images
samples = imgs[:3]
with zipfile.ZipFile(LOCAL, 'r') as z:
    for s in samples:
        z.extract(s, SAMPLE_DIR)

# Inspect each sample
print()
print('=== Sample image properties ===')
for s in samples:
    p = os.path.join(SAMPLE_DIR, s)
    try:
        im = Image.open(p)
        sz = os.path.getsize(p)
        print(f'  {s}')
        print(f'    Size: {sz/1024:.1f} KB, Mode: {im.mode}, Dimensions: {im.size}')
    except Exception as e:
        print(f'  {s}: error {e}')

## CELL 7 -- Summary dump

Prints a compact summary of everything above, ready to paste back.

In [ ]:
import zipfile
from pathlib import Path
from collections import Counter

LOCAL = '/content/M3OT.zip'
EXTRACT = Path('/content/m3ot_inspect')

print('=' * 60)
print('M3OT INSPECTION SUMMARY (paste this back to Claude)')
print('=' * 60)

with zipfile.ZipFile(LOCAL, 'r') as z:
    names = z.namelist()

print(f'\nZIP size: {Path(LOCAL).stat().st_size / 1e9:.2f} GB')
print(f'Total entries: {len(names)}')

# Extension breakdown
exts = Counter()
for n in names:
    if n.endswith('/'):
        continue
    e = n.split('.')[-1].lower() if '.' in n else 'noext'
    exts[e] += 1
print(f'\nFile types:')
for ext, c in exts.most_common(10):
    print(f'  .{ext}: {c}')

# Top 5 directory levels
print(f'\nTop-level dirs:')
top = set()
for n in names:
    top.add(n.split('/')[0])
for t in sorted(top):
    print(f'  {t}/')

# Extracted text files
print(f'\nText/config files found:')
text_files = list(EXTRACT.rglob('*.json')) + list(EXTRACT.rglob('*.yaml')) + list(EXTRACT.rglob('*.txt')) + list(EXTRACT.rglob('*.md'))
for f in sorted(text_files, key=lambda x: x.stat().st_size):
    print(f'  {f.stat().st_size/1024:>8.1f} KB  {f.relative_to(EXTRACT)}')

print()
print('Paste this full output + the output from Cells 3-6 back to Claude.')